# OpenAI o3 Deep-Research Evaluation

Evaluate a selected OpenAI deep-research model on Calculus I multiple-choice questions. Responses are validated locally and checkpointed after each question so interrupted runs can resume.

## 1. Imports and configuration

In [11]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)

### Run configuration

Change only `MODEL_NAME` to select another deep-research model. The artifact directory and every generated filename update automatically.

In [12]:
# Support launching Jupyter from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)

MODEL_NAME = "gpt-5.6-sol"
RUN_NAME = f"{MODEL_NAME}-o3-subsample"

DATA_PATH = PROJECT_ROOT / "data" / "test3_input_subset_seed42_numq1 (clean testing dataset).csv"
MODEL_OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / MODEL_NAME
CHECKPOINT_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-checkpoint.json"
RESULTS_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output.csv"
INCORRECT_RESULTS_PATH = (
    MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output-incorrect.csv"
)

MODEL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 2. OpenAI client and response schema

o3 deep research does not support API-enforced structured outputs. The model is instructed to return JSON, which is then parsed and validated locally with Pydantic.

In [13]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), timeout=3600)


class Calculus(BaseModel):
    """Validated response for one calculus question."""

    detailed_solution_explanation: str
    correct_option_choice_letter: str

### Few-shot example and instructions

In [14]:
EXAMPLE_RESPONSE = '''
"Complete the square to write the equation in the standard form f(x)=a(x-h)^2+k. f(x)=2x^2+12x+25
                a) f(x)=2(x+3)^2+16
                b) f(x)=2(x-6)^2+7
                c) f(x)=2(x+3)^2+7
                d) f(x)=2(x-3)^2+16
                e) f(x)=2(x+3)^2-7
                f) None of the above."
    "detailed_solution_explanation": "To complete the square for f(x) = 2x² + 12x + 25, first factor out 2 from the terms involving x:\n\nf(x) = 2(x² + 6x) + 25.\n\nNext, complete the square inside the parentheses:\n\nx² + 6x = x² + 6x + 9 − 9 = (x + 3)² − 9.\n\nRewrite f(x):\n\nf(x) = 2[(x + 3)² − 9] + 25\n      = 2(x + 3)² − 18 + 25\n      = 2(x + 3)² + 7.\n",
    "correct_option_choice_letter": "c"
'''

SYSTEM_INSTRUCTIONS = (
    "Act as a Mathematician specializing in Calculus. Solve the given "
    "college-level Calculus 1 multiple-choice problem, using the given example "
    "as a reference. Provide a step-by-step explanation of the solution process, "
    "clearly outlining the reasoning behind each step. Choose the correct letter "
    "option for the given problem. Ensure that the solution is accurate, "
    "comprehensive, and easy to understand. Return only one valid JSON object "
    "with the keys detailed_solution_explanation and "
    "correct_option_choice_letter. Do not wrap the JSON in Markdown. "
    "Here is the example: "
    + EXAMPLE_RESPONSE
)

## 3. Load the dataset

In [15]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the evaluation dataset."""
    return pd.read_csv(data_path)


df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} questions from {DATA_PATH}")
df.head()

Loaded 331 questions from /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/data/test3_input_subset_seed42_numq1 (clean testing dataset).csv


,ID,Question,Correct Answer
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c


## 4. Response pipeline

Each successfully validated response is written to the model-specific checkpoint immediately.

In [16]:
def load_checkpoint(checkpoint_path: Path) -> dict:
    """Load saved responses, or return an empty checkpoint."""
    if not checkpoint_path.exists():
        return {}
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint_file:
        return json.load(checkpoint_file)


def save_checkpoint(checkpoint_path: Path, data: dict) -> None:
    """Persist responses as formatted JSON."""
    with checkpoint_path.open("w", encoding="utf-8") as checkpoint_file:
        json.dump(data, checkpoint_file, indent=4)

In [17]:
def parse_calculus_response(response_text: str) -> dict:
    """Extract and validate the JSON object returned by the model."""
    start = response_text.find("{")
    end = response_text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("The model response did not contain a JSON object.")

    payload = json.loads(response_text[start : end + 1])
    return Calculus.model_validate(payload).model_dump()


def query_openai(question: str, model: str = MODEL_NAME) -> dict:
    """Request and locally validate one deep-research calculus solution."""
    if not hasattr(client, "responses"):
        raise RuntimeError(
            "This notebook requires an OpenAI Python SDK version that supports "
            "client.responses. Upgrade the openai package and restart the kernel."
        )

    response = client.responses.create(
        model=model,
        instructions=SYSTEM_INSTRUCTIONS,
        input="Now solve the following question: " + question,
        tools=[{"type": "web_search_preview"}],
    )
    return parse_calculus_response(response.output_text)

In [18]:
def collect_responses(
    dataframe: pd.DataFrame,
    checkpoint_path: Path,
    model: str = MODEL_NAME,
) -> dict:
    """Query unanswered rows and update the checkpoint incrementally."""
    checkpoint = load_checkpoint(checkpoint_path)

    for _, row in dataframe.iterrows():
        question_id = str(row["ID"])
        if question_id in checkpoint:
            print(f"Skipping ID {question_id}, already processed.")
            continue

        print(f"Processing ID {question_id}...")
        response = query_openai(row["Question"], model=model)
        checkpoint[question_id] = response
        save_checkpoint(checkpoint_path, checkpoint)

    save_checkpoint(checkpoint_path, checkpoint)
    print(f"Checkpoint saved to {checkpoint_path}")
    return checkpoint

In [19]:
def add_model_responses(dataframe: pd.DataFrame, checkpoint: dict) -> pd.DataFrame:
    """Add answer choices and explanations from the checkpoint."""
    result = dataframe.copy()
    question_ids = result["ID"].astype(str)
    result["LLM Answer Choice (RAW)"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "correct_option_choice_letter", ""
        )
    )
    result["LLM Answer Explanation"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "detailed_solution_explanation", ""
        )
    )
    return result

### Run the response pipeline

> **Note:** This cell makes billable deep-research API calls only for questions absent from the checkpoint. Deep-research requests can be slow and comparatively expensive.

In [20]:
df = df.head(10)  # Limit to first 10 rows for testing

checkpoint = collect_responses(df, CHECKPOINT_PATH)
df = add_model_responses(df, checkpoint)
df.head(10)

Processing ID q2_sp22_problem10__04...
Processing ID q27_sp23_problem04__04...
Processing ID q23_sp23_problem09__02...
Processing ID q31_sp23_problem03__09...
Processing ID q10_sp22_problem07__21...
Processing ID q19_fa22_problem03__21...
Processing ID q28_sp23_problem03__04...
Processing ID q25_sp23_problem08__02...
Processing ID q2_sp22_problem06__04...
Processing ID q5_sp22_problem07__01...
Checkpoint saved to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/gpt-5.6-sol/gpt-5.6-sol-o3-subsample-checkpoint.json


,ID,Question,Correct Answer,LLM Answer Choice (RAW),LLM Answer Explanation
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,b,To complete the square for f(x) = 4x² − 48x + ...
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,b,Let x be the side perpendicular to the dividin...
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,b,Differentiate f(x) = tan(5^(6x²)) using the ch...
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,d,"Given\n\nF(x) = ∫₀ˣ √t(t + 1)² dt,\n\napply th..."
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,c,The slope of the tangent line to the graph of ...
5,q19_fa22_problem03__21,21) Find the intervals on which \(f(x)=\displa...,c,c,"To find where f(x) = 6x/(x² + 4) decreases, de..."
6,q28_sp23_problem03__04,4) Use differentials to estimate the value of ...,e,e,"Let f(x) = x^(1/4). To estimate (80.5)^(1/4), ..."
7,q25_sp23_problem08__02,2) Differentiate the following: \(f(x)=\displ...,b,b,Given\n\nf(x) = ln(arctan(5x + 6)).\n\nThis is...
8,q2_sp22_problem06__04,4) Find the <i>x</i>- and <i>y</i>-intercepts ...,b,b,"To find the x-intercepts, set f(x) equal to 0:..."
9,q5_sp22_problem07__01,1) Evaluate \(\displaystyle \lim _{x\rightarro...,c,c,To evaluate the two-sided limit as x approache...


## 5. Normalize and evaluate answers

In [21]:
def extract_letter_choice(text: str) -> str:
    """Return the first alphabetic character from a raw answer choice."""
    match = re.search(r"[a-zA-Z]", text)
    return match.group(0) if match else ""


def evaluate_answers(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Normalize predictions and calculate weighted classification metrics."""
    result = dataframe.copy()
    result["LLM Answer Choice"] = result["LLM Answer Choice (RAW)"].map(
        extract_letter_choice
    )
    result["Correct"] = (
        result["Correct Answer"] == result["LLM Answer Choice"]
    ).astype(int)

    y_true = result["Correct Answer"]
    y_pred = result["LLM Answer Choice"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "classification_report": classification_report(
            y_true, y_pred, zero_division=0
        ),
    }
    return result, metrics

In [22]:
df, metrics = evaluate_answers(df)

print(metrics["classification_report"])
print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"Precision: {metrics['precision'] * 100:.2f}%")
print(f"Recall:    {metrics['recall'] * 100:.2f}%")
print(f"F1 Score:  {metrics['f1'] * 100:.2f}%")
df.head()

              precision    recall  f1-score   support

           b       1.00      1.00      1.00         5
           c       1.00      1.00      1.00         3
           d       1.00      1.00      1.00         1
           e       1.00      1.00      1.00         1

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

Accuracy:  100.00%
Precision: 100.00%
Recall:    100.00%
F1 Score:  100.00%


,ID,Question,Correct Answer,LLM Answer Choice (RAW),LLM Answer Explanation,LLM Answer Choice,Correct
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,b,To complete the square for f(x) = 4x² − 48x + ...,b,1
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,b,Let x be the side perpendicular to the dividin...,b,1
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,b,Differentiate f(x) = tan(5^(6x²)) using the ch...,b,1
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,d,"Given\n\nF(x) = ∫₀ˣ √t(t + 1)² dt,\n\napply th...",d,1
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,c,The slope of the tangent line to the graph of ...,c,1


## 6. Export results

In [23]:
incorrect_df = df.loc[df["Correct"] == 0].copy()

incorrect_df.to_csv(INCORRECT_RESULTS_PATH, index=False)
df.to_csv(RESULTS_PATH, index=False)

print(f"Saved {len(df)} results to {RESULTS_PATH}")
print(f"Saved {len(incorrect_df)} incorrect results to {INCORRECT_RESULTS_PATH}")

Saved 10 results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/gpt-5.6-sol/gpt-5.6-sol-o3-subsample-test-output.csv
Saved 0 incorrect results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/Publishable Project Repo/output/gpt-5.6-sol/gpt-5.6-sol-o3-subsample-test-output-incorrect.csv
